# ChurnGuard API — Live Tests
Testing the deployed Render API: `https://customer-churn-prediction-02k2.onrender.com`

In [ ]:
import requests

BASE = "https://customer-churn-prediction-02k2.onrender.com"

passed, failed = [], []

def check(name, condition, detail=""):
    if condition:
        passed.append(name)
        print(f"  PASS  {name}")
    else:
        failed.append(name)
        print(f"  FAIL  {name}" + (f" — {detail}" if detail else ""))

BANK_PAYLOAD = {
    "name": "Jane Smith", "credit_score": 650, "geography": "France",
    "gender": "Female", "age": 42, "tenure": 5, "balance": 75000.0,
    "num_products": 2, "has_cr_card": 1, "is_active_member": 1, "estimated_salary": 85000.0,
}

TELCO_PAYLOAD = {
    "name": "John Rivera", "gender": "Male", "senior_citizen": 0,
    "partner": "No", "dependents": "No", "tenure": 12,
    "phone_service": "Yes", "multiple_lines": "No",
    "internet_service": "Fiber optic", "online_security": "No",
    "online_backup": "No", "device_protection": "No", "tech_support": "No",
    "streaming_tv": "Yes", "streaming_movies": "Yes",
    "contract": "Month-to-month", "paperless_billing": "Yes",
    "payment_method": "Electronic check",
    "monthly_charges": 89.85, "total_charges": 1078.20,
}

print("Setup complete.")

: 

## Meta

In [ ]:
r = requests.get(f"{BASE}/health", timeout=60)
b = r.json()
check("GET /health — status 200",       r.status_code == 200)
check("GET /health — status ok",        b.get("status") == "ok")
check("GET /health — 7 bank models",    b.get("bank_models") == 7)
check("GET /health — 5 telco models",   b.get("telco_models") == 5)
print()

r = requests.get(f"{BASE}/models", timeout=60)
b = r.json()
check("GET /models — status 200",       r.status_code == 200)
check("GET /models — 7 bank models",    len(b.get("bank", [])) == 7)
check("GET /models — 5 telco models",   len(b.get("telco", [])) == 5)
print("\nBank models: ",  b.get("bank"))
print("Telco models:", b.get("telco"))

## Bank Churn

In [ ]:
r = requests.post(f"{BASE}/predict/bank", json=BANK_PAYLOAD, timeout=60)
b = r.json()
prob = b.get("churn_probability", -1)

check("POST /predict/bank — status 200",          r.status_code == 200)
check("POST /predict/bank — probability in [0,1]", 0.0 <= prob <= 1.0)
check("POST /predict/bank — risk_level valid",     b.get("risk_level") in {"low", "medium", "high"})
check("POST /predict/bank — 7 model scores",       len(b.get("model_scores", {})) == 7)
check("POST /predict/bank — scores in [0,1]",      all(0 <= v <= 1 for v in b.get("model_scores", {}).values()))
check("POST /predict/bank — dataset label",        b.get("dataset") == "bank")
print(f"\nChurn probability: {prob:.1%}  |  Risk: {b.get('risk_level')}")
print("Model scores:")
for model, score in b.get("model_scores", {}).items():
    print(f"  {model:<22} {score:.4f}")

In [ ]:
# High-risk profile
high_risk = {**BANK_PAYLOAD, "age": 55, "balance": 0, "is_active_member": 0, "num_products": 4}
r = requests.post(f"{BASE}/predict/bank", json=high_risk, timeout=60)
prob = r.json().get("churn_probability", 0)
check("POST /predict/bank — high-risk profile > 0.4", prob > 0.4, f"got {prob:.3f}")

# Validation errors
r = requests.post(f"{BASE}/predict/bank", json={**BANK_PAYLOAD, "credit_score": 100}, timeout=60)
check("POST /predict/bank — credit_score 100 → 422", r.status_code == 422)

r = requests.post(f"{BASE}/predict/bank", json={**BANK_PAYLOAD, "gender": "Unknown"}, timeout=60)
check("POST /predict/bank — invalid gender → 422",   r.status_code == 422)

## Telco Churn

In [ ]:
r = requests.post(f"{BASE}/predict/telco", json=TELCO_PAYLOAD, timeout=60)
b = r.json()
prob = b.get("churn_probability", -1)

check("POST /predict/telco — status 200",           r.status_code == 200)
check("POST /predict/telco — probability in [0,1]", 0.0 <= prob <= 1.0)
check("POST /predict/telco — risk_level valid",     b.get("risk_level") in {"low", "medium", "high"})
check("POST /predict/telco — 5 model scores",       len(b.get("model_scores", {})) == 5)
check("POST /predict/telco — scores in [0,1]",      all(0 <= v <= 1 for v in b.get("model_scores", {}).values()))
check("POST /predict/telco — dataset label",        b.get("dataset") == "telco")
print(f"\nChurn probability: {prob:.1%}  |  Risk: {b.get('risk_level')}")
print("Model scores:")
for model, score in b.get("model_scores", {}).items():
    print(f"  {model:<22} {score:.4f}")

In [ ]:
# Low-risk profile
low_risk = {**TELCO_PAYLOAD, "contract": "Two year", "tenure": 60, "monthly_charges": 30.0}
r = requests.post(f"{BASE}/predict/telco", json=low_risk, timeout=60)
prob = r.json().get("churn_probability", 1)
check("POST /predict/telco — low-risk profile < 0.6", prob < 0.6, f"got {prob:.3f}")

# Validation error
r = requests.post(f"{BASE}/predict/telco", json={**TELCO_PAYLOAD, "contract": "Weekly"}, timeout=60)
check("POST /predict/telco — invalid contract → 422", r.status_code == 422)

## Summary

In [ ]:
total = len(passed) + len(failed)
print(f"Results: {len(passed)}/{total} passed")
if failed:
    print("\nFailed:")
    for name in failed:
        print(f"  FAIL  {name}")